# Byte-Pair Encoding: How Text Becomes Tokens, One Merge at a Time

The full code of [Byte-Pair Encoding: How Text Becomes Tokens, One Merge at a Time](https://residual-stream.lvtan71.workers.dev/posts/bpe-tokenizer-explained/), run from [its folder](https://github.com/lvtan71/residual-stream-code/tree/main/posts/bpe-tokenizer-explained) in the
order the folder's README gives. Each script is followed by the output it
printed for the post, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/bpe-tokenizer-explained.ipynb)

**Hardware:** Runs on a CPU; Colab's free runtime is enough. About 5 min end to end on the RTX 3090 machine that saved the outputs below.
Trains byte-level BPE on TinyStories' validation and training splits; the training split takes the longest.

The post's numbers come from an Apple M4 laptop. Other hardware and library
versions change timings and the last digits of some floating-point results.
Sampled text and small training runs can move further; where that changes a
number the post discusses, the note above says so.

**Outputs saved from a run on NVIDIA GeForce RTX 3090 on 2026-10-18** (torch 2.14.1+cu130, 5 min; every cell ran). Run the notebook to replace them with your own.

In [1]:
%pip install -q regex tiktoken huggingface_hub pandas pyarrow

# Setup: fetch the companion repo and work in this post's folder.
import os, subprocess, sys
SLUG = "bpe-tokenizer-explained"
ROOT = os.path.abspath("..") if os.path.isdir("../posts") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
os.chdir(os.path.join(ROOT, "posts", SLUG))
print("working in", os.getcwd())

Note: you may need to restart the kernel to use updated packages.
working in residual-stream-code/posts/bpe-tokenizer-explained


## `toy.py`

Trains on the 16-word corpus from Sennrich et al. and prints every table, merge and encoding step.

In [2]:
!{sys.executable} toy.py

pre-tokens, in order: low ␣low ␣low ␣low ␣low ⏎ lower ␣lower ␣widest ␣widest ␣widest ⏎ newest ␣newest ␣newest ␣newest ␣newest ␣newest ⏎

8 distinct pre-tokens, 19 occurrences
  (␣, n, e, w, e, s, t)    5
  (␣, l, o, w)             4
  (⏎)                      3
  (␣, w, i, d, e, s, t)    3
  (␣, l, o, w, e, r)       1
  (l, o, w)                1
  (l, o, w, e, r)          1
  (n, e, w, e, s, t)       1

14 distinct adjacent pairs
  (e, s)       9
  (s, t)       9
  (w, e)       8
  (l, o)       7
  (o, w)       7
  (e, w)       6
  (n, e)       6
  (␣, l)       5
  (␣, n)       5
  (␣, w)       3
  (d, e)       3
  (i, d)       3
  (w, i)       3
  (e, r)       2

 #   id  pairs at the top count         merge         new token  count
 1  256  (e, s) (s, t)                 s + t         st         9
 2  257  (e, st)                       e + st        est        9
 3  258  (l, o) (o, w)                 o + w         ow         7
 4  259  (l, ow)                       l + ow        low 

The post printed:

```text
pre-tokens, in order: low ␣low ␣low ␣low ␣low ⏎ lower ␣lower ␣widest ␣widest ␣widest ⏎ newest ␣newest ␣newest ␣newest ␣newest ␣newest ⏎

8 distinct pre-tokens, 19 occurrences
  (␣, n, e, w, e, s, t)    5
  (␣, l, o, w)             4
  (⏎)                      3
  (␣, w, i, d, e, s, t)    3
  (␣, l, o, w, e, r)       1
  (l, o, w)                1
  (l, o, w, e, r)          1
  (n, e, w, e, s, t)       1

14 distinct adjacent pairs
  (e, s)       9
  (s, t)       9
  (w, e)       8
  (l, o)       7
  (o, w)       7
  (e, w)       6
  (n, e)       6
  (␣, l)       5
  (␣, n)       5
  (␣, w)       3
  (d, e)       3
  (i, d)       3
  (w, i)       3
  (e, r)       2

 #   id  pairs at the top count         merge         new token  count
 1  256  (e, s) (s, t)                 s + t         st         9
 2  257  (e, st)                       e + st        est        9
 3  258  (l, o) (o, w)                 o + w         ow         7
 4  259  (l, ow)                       l + ow        low        7
 5  260  (e, w) (n, e) (w, est)        w + est       west       6
 6  261  (e, west) (n, e)              n + e         ne         6
 7  262  (ne, west)                    ne + west      newest     6
 8  263  (␣, low) (␣, newest)          ␣ + newest    ␣newest    5

vocab beyond the bytes: {256: b'st', 257: b'est', 258: b'ow', 259: b'low', 260: b'west', 261: b'ne', 262: b'newest', 263: b' newest', 264: b'<|endoftext|>'}
merges: [(b's', b't'), (b'e', b'st'), (b'o', b'w'), (b'l', b'ow'), (b'w', b'est'), (b'n', b'e'), (b'ne', b'west'), (b' ', b'newest')]

encode '␣newest'
  start:  (␣, n, e, w, e, s, t)
  iter 1: (n, e)#5 (s, t)#0                  -> apply rank 0 (s, t)
  iter 2: (n, e)#5 (e, st)#1                 -> apply rank 1 (e, st)
  iter 3: (n, e)#5 (w, est)#4                -> apply rank 4 (w, est)
  iter 4: (n, e)#5                           -> apply rank 5 (n, e)
  iter 5: (ne, west)#6                       -> apply rank 6 (ne, west)
  iter 6: (␣, newest)#7                      -> apply rank 7 (␣, newest)
  result: (␣newest)  ids [263]

encode 'lowest'
  start:  (l, o, w, e, s, t)
  iter 1: (o, w)#2 (s, t)#0                  -> apply rank 0 (s, t)
  iter 2: (o, w)#2 (e, st)#1                 -> apply rank 1 (e, st)
  iter 3: (o, w)#2 (w, est)#4                -> apply rank 2 (o, w)
  iter 4: (l, ow)#3                          -> apply rank 3 (l, ow)
  result: (low, est)  ids [259, 257]

newer: (ne, w, e, r)  ids [261, 119, 101, 114]

zzqq: (z, z, q, q)  ids [122, 122, 113, 113]

text:  'the lowest widest fox<|endoftext|>newest lower'
split: ['the lowest widest fox', '<|endoftext|>', 'newest lower']
ids:   [116, 104, 101, 32, 259, 257, 32, 119, 105, 100, 257, 32, 102, 111, 120, 264, 262, 32, 259, 101, 114]
tokens: t | h | e | ␣ | low | est | ␣ | w | i | d | est | ␣ | f | o | x | <|endoftext|> | newest | ␣ | low | e | r
46 bytes -> 21 tokens = 2.19 bytes per token
round trip: True

'🙃' -> ids [240, 159, 153, 131] -> '🙃'
first two bytes only [240, 159] -> '�'
```

## `utf8.py`

Characters as UTF-8 bytes, and every code point checked for bytes below 0x80 inside a multi-byte character.

In [3]:
!{sys.executable} utf8.py

s  U+0073  1 byte: 73           = [115]
é  U+00E9  2 bytes: C3 A9        = [195, 169]
à  U+00E0  2 bytes: C3 A0        = [195, 160]
牛  U+725B  3 bytes: E7 89 9B     = [231, 137, 155]
🙃  U+1F643  4 bytes: F0 9F 99 83  = [240, 159, 153, 131]
'héllo': 5 characters, 6 bytes [104, 195, 169, 108, 108, 111]
'Xin chào': 8 characters, 9 bytes [88, 105, 110, 32, 99, 104, 195, 160, 111]



code points by UTF-8 length: 1 byte: 128, 2 bytes: 1,920, 3 bytes: 61,440, 4 bytes: 1,048,576
total 1,112,064; bytes below 0x80 inside a multi-byte character: 0


The post printed:

```text
s  U+0073  1 byte: 73           = [115]
é  U+00E9  2 bytes: C3 A9        = [195, 169]
à  U+00E0  2 bytes: C3 A0        = [195, 160]
牛  U+725B  3 bytes: E7 89 9B     = [231, 137, 155]
🙃  U+1F643  4 bytes: F0 9F 99 83  = [240, 159, 153, 131]
'héllo': 5 characters, 6 bytes [104, 195, 169, 108, 108, 111]
'Xin chào': 8 characters, 9 bytes [88, 105, 110, 32, 99, 104, 195, 160, 111]

code points by UTF-8 length: 1 byte: 128, 2 bytes: 1,920, 3 bytes: 61,440, 4 bytes: 1,048,576
total 1,112,064; bytes below 0x80 inside a multi-byte character: 0
```

## `tiktoken_compare.py`

OpenAI's GPT-2, cl100k_base and o200k_base tokenizers on the post's examples.

In [4]:
!{sys.executable} tiktoken_compare.py

gpt2         vocabulary 50,257


cl100k_base  vocabulary 100,277


o200k_base   vocabulary 200,019

gpt2 'accomplishment': 3 tokens  ac | compl | ishment

gpt2 ' accomplishment': 1 token  ␣accomplishment

gpt2 'the lowest widest fox newest lower': 6 tokens  the | ␣lowest | ␣widest | ␣fox | ␣newest | ␣lower

'12345678': 8 characters, 8 bytes
  gpt2          3 tokens  123 | 45 | 678
  cl100k_base   3 tokens  123 | 456 | 78
  o200k_base    3 tokens  123 | 456 | 78

'Xin chào thế giới': 17 characters, 22 bytes
  gpt2         15 tokens  X | in | ␣ch | à | o | ␣th | <E1> | <BA> | <BF> | ␣g | i | <E1> | <BB> | <9B> | i
  cl100k_base   9 tokens  X | in | ␣ch | à | o | ␣th | ế | ␣gi | ới
  o200k_base    5 tokens  Xin | ␣ch | ào | ␣thế | ␣giới


The post printed:

```text
gpt2         vocabulary 50,257
cl100k_base  vocabulary 100,277
o200k_base   vocabulary 200,019

gpt2 'accomplishment': 3 tokens  ac | compl | ishment

gpt2 ' accomplishment': 1 token  ␣accomplishment

gpt2 'the lowest widest fox newest lower': 6 tokens  the | ␣lowest | ␣widest | ␣fox | ␣newest | ␣lower

'12345678': 8 characters, 8 bytes
  gpt2          3 tokens  123 | 45 | 678
  cl100k_base   3 tokens  123 | 456 | 78
  o200k_base    3 tokens  123 | 456 | 78

'Xin chào thế giới': 17 characters, 22 bytes
  gpt2         15 tokens  X | in | ␣ch | à | o | ␣th | <E1> | <BA> | <BF> | ␣g | i | <E1> | <BB> | <9B> | i
  cl100k_base   9 tokens  X | in | ␣ch | à | o | ␣th | ế | ␣gi | ới
  o200k_base    5 tokens  Xin | ␣ch | ào | ␣thế | ␣giới
```

## `files.py`

GPT-2's byte-to-character table, and saving and loading `vocab.json` and `merges.txt`.

In [5]:
!{sys.executable} files.py

bytes kept as themselves: 188, moved: 68
space -> Ġ  newline -> Ċ  byte 0 -> Ā

toy_vocab.json, beyond the bytes: {'st': 256, 'est': 257, 'ow': 258, 'low': 259, 'west': 260, 'ne': 261, 'newest': 262, 'Ġnewest': 263, '<|endoftext|>': 264}
toy_merges.txt:
s t
e st
o w
l ow
w est
n e
ne west
Ġ newest

load(save(...)) gives back the same tokenizer: True


The post printed:

```text
bytes kept as themselves: 188, moved: 68
space -> Ġ  newline -> Ċ  byte 0 -> Ā

toy_vocab.json, beyond the bytes: {'st': 256, 'est': 257, 'ow': 258, 'low': 259, 'west': 260, 'ne': 261, 'newest': 262, 'Ġnewest': 263, '<|endoftext|>': 264}
toy_merges.txt:
s t
e st
o w
l ow
w est
n e
ne west
Ġ newest

load(save(...)) gives back the same tokenizer: True
```

## `data.py`

Downloads TinyStories (2.2 GB) and the OPUS-100 en-vi test split at pinned revisions.

In [6]:
!{sys.executable} data.py

TinyStoriesV2-GPT4-valid.txt                  22,502,601 bytes, 27,630 <|endoftext|> separators


TinyStoriesV2-GPT4-train.txt               2,227,753,162 bytes, 2,717,699 <|endoftext|> separators


test-00000-of-00001.parquet                      137,302 bytes


The post printed:

```text
TinyStoriesV2-GPT4-valid.txt                  22,502,601 bytes, 27,630 <|endoftext|> separators
TinyStoriesV2-GPT4-train.txt               2,227,753,162 bytes, 2,717,699 <|endoftext|> separators
test-00000-of-00001.parquet                      137,302 bytes
```

## `check.py`

Checks the fast trainer against the naive one and the heap against the scan, and that 1, 2, 4 and 8 workers count the same pre-tokens.

In [7]:
!{sys.executable} check.py

first 500,000 characters, 500 merges: indexed == naive: True, heap == naive: True


whole valid file, pre-token tables at 1, 2, 4, 8 workers identical: True (13,111 distinct pre-tokens)


whole valid file, 10,000-token vocabulary: heap == scan: True


The post printed:

```text
first 500,000 characters, 500 merges: indexed == naive: True, heap == naive: True
whole valid file, pre-token tables at 1, 2, 4, 8 workers identical: True (13,111 distinct pre-tokens)
whole valid file, 10,000-token vocabulary: heap == scan: True
```

## `no_regex.py`

The first 20 merges on the first million characters of TinyStories with and without the pre-tokenizer.

In [8]:
!{sys.executable} no_regex.py

no pre-tokenizer:  e␣  d␣  th  ␣a  .␣  t␣  y␣  s␣  nd␣  to  er  ed␣  the␣  ,␣  wa  in  he␣  to␣  ␣and␣  ou


GPT-2's regex:     he  ␣t  ␣a  ␣s  ␣w  ␣the  nd  ed  ␣b  ␣to  ␣and  ␣h  ␣T  ␣f  ␣wa  in  it  re  ou  ␣l


The post printed:

```text
no pre-tokenizer:  e␣  d␣  th  ␣a  .␣  t␣  y␣  s␣  nd␣  to  er  ed␣  the␣  ,␣  wa  in  he␣  to␣  ␣and␣  ou
GPT-2's regex:     he  ␣t  ␣a  ␣s  ␣w  ␣the  nd  ed  ␣b  ␣to  ␣and  ␣h  ␣T  ␣f  ␣wa  in  it  re  ou  ␣l
```

## `tinystories.py valid`

Trains a 10,000-token vocabulary on `valid` (22 MB) or `train` (2.2 GB), timing each stage with a scan and with a heap, and writes `ts_<split>_vocab.json` and `ts_<split>_merges.txt`.

In [9]:
!{sys.executable} tinystories.py valid

TinyStoriesV2-GPT4-valid.txt: 22,502,601 bytes; 8 workers; x86_64, 256 cores, Python 3.12.3


pre-tokenize:    1.5 s  (13,111 distinct pre-tokens, 5,419,001 in all)


scan: pick the pair   13.7 s, apply the merge   0.6 s; with pre-tokenizing   15.8 s (pre-tokenize 9%, pick 87%, apply 4%)


heap: pick the pair    0.2 s, apply the merge   0.7 s; with pre-tokenizing    2.4 s (pre-tokenize 62%, pick 10%, apply 28%)
9,743 merges; vocabulary 10,000; <|endoftext|> is ID 9,999
peak memory: 62 MB in the main process, 39 MB in the largest worker


The post printed:

```text
TinyStoriesV2-GPT4-valid.txt: 22,502,601 bytes; 8 workers; arm64, 10 cores, Python 3.12.13
pre-tokenize:    0.8 s  (13,111 distinct pre-tokens, 5,419,001 in all)
scan: pick the pair    6.0 s, apply the merge   0.3 s; with pre-tokenizing    7.1 s (pre-tokenize 12%, pick 84%, apply 4%)
heap: pick the pair    0.1 s, apply the merge   0.3 s; with pre-tokenizing    1.3 s (pre-tokenize 66%, pick 10%, apply 24%)
9,743 merges; vocabulary 10,000; <|endoftext|> is ID 9,999
peak memory: 77 MB in the main process, 58 MB in the largest worker
```

## `tinystories.py train`

Trains a 10,000-token vocabulary on `valid` (22 MB) or `train` (2.2 GB), timing each stage with a scan and with a heap, and writes `ts_<split>_vocab.json` and `ts_<split>_merges.txt`.

In [10]:
!{sys.executable} tinystories.py train

TinyStoriesV2-GPT4-train.txt: 2,227,753,162 bytes; 8 workers; x86_64, 256 cores, Python 3.12.3


pre-tokenize:  132.7 s  (59,933 distinct pre-tokens, 536,592,168 in all)


scan: pick the pair   62.0 s, apply the merge   3.5 s; with pre-tokenizing  198.2 s (pre-tokenize 67%, pick 31%, apply 2%)


heap: pick the pair    0.7 s, apply the merge   4.5 s; with pre-tokenizing  137.9 s (pre-tokenize 96%, pick 1%, apply 3%)
9,743 merges; vocabulary 10,000; <|endoftext|> is ID 9,999
peak memory: 142 MB in the main process, 1,525 MB in the largest worker


The post printed:

```text
TinyStoriesV2-GPT4-train.txt: 2,227,753,162 bytes; 8 workers; arm64, 10 cores, Python 3.12.13
pre-tokenize:   85.5 s  (59,933 distinct pre-tokens, 536,592,168 in all)
scan: pick the pair   24.7 s, apply the merge   1.3 s; with pre-tokenizing  111.5 s (pre-tokenize 77%, pick 22%, apply 1%)
heap: pick the pair    0.5 s, apply the merge   2.2 s; with pre-tokenizing   88.2 s (pre-tokenize 97%, pick 1%, apply 2%)
9,743 merges; vocabulary 10,000; <|endoftext|> is ID 9,999
peak memory: 161 MB in the main process, 769 MB in the largest worker
```

## `analyze.py`

What the two TinyStories tokenizers learned, and their compression on 200 held-out stories against GPT-2's.

In [11]:
!{sys.executable} analyze.py

200 held-out stories, 156,695 bytes
  this tokenizer (10,000):  38,337 tokens, 4.09 bytes per token
  GPT-2's (50,257):         38,775 tokens, 4.04 bytes per token
  round trip exact: 200 of 200

first 12 merges: ␣t  he  ␣a  ␣s  ␣w  nd  ␣the  ed  ␣b  ␣to  ␣and  ␣h
ten longest tokens: ␣accomplishment, ␣disappointment, ␣responsibility, ␣Unfortunately, ␣compassionate, ␣congratulated, ␣determination, ␣encouragement, ␣extraordinary, ␣granddaughter
7,277 of 9,743 learned tokens start with a space (75%)
learned tokens by length in bytes: 2: 360, 3: 912, 4: 1450, 5: 1835, 6: 1666, 7: 1264, 8: 1030, 9: 602, 10: 328, 11: 178, 12: 78, 13: 26, 14: 11, 15: 3

'Once upon a time, there was a little boat.'
  this tokenizer: Once | ␣upon | ␣a | ␣time | , | ␣there | ␣was | ␣a | ␣little | ␣boat | .
  GPT-2:          Once | ␣upon | ␣a | ␣time | , | ␣there | ␣was | ␣a | ␣little | ␣boat | .

trained on 1% of the data (the 22 MB validation file): 8,825 of 10,000 vocabulary entries shared
merge lists first di

The post printed:

```text
200 held-out stories, 156,695 bytes
  this tokenizer (10,000):  38,337 tokens, 4.09 bytes per token
  GPT-2's (50,257):         38,775 tokens, 4.04 bytes per token
  round trip exact: 200 of 200

first 12 merges: ␣t  he  ␣a  ␣s  ␣w  nd  ␣the  ed  ␣b  ␣to  ␣and  ␣h
ten longest tokens: ␣accomplishment, ␣disappointment, ␣responsibility, ␣Unfortunately, ␣compassionate, ␣congratulated, ␣determination, ␣encouragement, ␣extraordinary, ␣granddaughter
7,277 of 9,743 learned tokens start with a space (75%)
learned tokens by length in bytes: 2: 360, 3: 912, 4: 1450, 5: 1835, 6: 1666, 7: 1264, 8: 1030, 9: 602, 10: 328, 11: 178, 12: 78, 13: 26, 14: 11, 15: 3

'Once upon a time, there was a little boat.'
  this tokenizer: Once | ␣upon | ␣a | ␣time | , | ␣there | ␣was | ␣a | ␣little | ␣boat | .
  GPT-2:          Once | ␣upon | ␣a | ␣time | , | ␣there | ␣was | ␣a | ␣little | ␣boat | .

trained on 1% of the data (the 22 MB validation file): 8,825 of 10,000 vocabulary entries shared
merge lists first differ at merge 6: n + d (full data) vs ␣t + he (1%)
```

## `languages.py`

2,000 English-Vietnamese sentence pairs counted with four tokenizers.

In [12]:
!{sys.executable} languages.py

2,000 sentence pairs
            characters  UTF-8 bytes    words
English         80,240       80,282   15,183
Vietnamese      77,721      100,460   17,437



tokenizer     English bytes/tok  Vietnamese bytes/tok  Vietnamese / English


gpt2           22,689      3.54      66,891      1.50                 2.95x
cl100k_base    20,756      3.87      39,686      2.53                 1.91x
o200k_base     19,947      4.02      25,470      3.94                 1.28x


TinyStories    25,938      3.10      76,892      1.31                 2.96x


The post printed:

```text
2,000 sentence pairs
            characters  UTF-8 bytes    words
English         80,240       80,282   15,183
Vietnamese      77,721      100,460   17,437

tokenizer     English bytes/tok  Vietnamese bytes/tok  Vietnamese / English
gpt2           22,689      3.54      66,891      1.50                 2.95x
cl100k_base    20,756      3.87      39,686      2.53                 1.91x
o200k_base     19,947      4.02      25,470      3.94                 1.28x
TinyStories    25,938      3.10      76,892      1.31                 2.96x
```